<a href="https://colab.research.google.com/github/MadhuryaPasan/SLIIT-MLOM-Project-about-LoRA/blob/main/LoRA_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"]= userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"]= userdata.get('KAGGLE_KEY')

In [ ]:
!pip install -q -U keras-nlp
!pip install -U keras-hub keras

In [ ]:
# slect a backend
os.environ["KERAS_BACKEND"] = "jax" # or torch ot tensorflow
# Avoid memory fragmentation on JAX backend
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00"

In [ ]:
# import packages
import keras
# import keras_nlp
import keras_hub

In [ ]:
!wget -O databricks-dolly-15k.jsonl https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl

In [ ]:
import json

prompts, responses = [], []

with open("databricks-dolly-15k.jsonl") as file:
    for line in file:
        features = json.loads(line)
        # Filter out examples with context
        if features["context"]:
            continue
        prompts.append(features["instruction"])
        responses.append(features["response"])

# Use only first 1000 examples for faster training
prompts, responses = prompts[:1000], responses[:1000]

features = {"prompts": prompts, "responses": responses}

In [ ]:
# Quick check
print("Sample prompt:", features["prompts"][0])
print("Sample response:", features["responses"][0])

In [ ]:
#Load the pretrained gemma model - Gemma3CausalLM → a causal language model (for text generation).
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_1b")

#Show model architecture
gemma_lm.summary()

In [ ]:
#Create a sampling strategy
sampler = keras_hub.samplers.TopKSampler(k=40, seed=42, temperature=0.8)

#Compile the model with the sampler
gemma_lm.compile(sampler=sampler)

In [ ]:
#Define a prompt
prompt = "Instruction:\nWrite a short story about a person who discovers a hidden room in their house.\n\nResponse:\n"

#Generate text
output = gemma_lm.generate(prompt, max_length=128)

#Print the generated story
print("\n", output)

In [ ]:
prompt = "Instruction:\nWhat should I do on a trip to Europe?\n\nResponse:\n"
output = gemma_lm.generate(prompt, max_length=128)
print("\n", output)

In [ ]:
#Enable LoRA fine-tuning
gemma_lm.backbone.enable_lora(rank=16)

#Show new model structure
gemma_lm.summary()

In [ ]:
gemma_lm.preprocessor.sequence_length = 256
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)
optimizer.exclude_from_weight_decay(["bias", "scale"])

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

gemma_lm.fit(x=features, epochs=1, batch_size=8)

# After LoRA

In [ ]:
prompt = "Instruction:\nWrite a short story about a person who discovers a hidden room in their house.\n\nResponse:\n"
output = gemma_lm.generate(prompt, max_length=128)
print("\n", output)

In [ ]:
prompt = "Instruction:\nWhat should I do on a trip to Europe?\n\nResponse:\n"
output = gemma_lm.generate(prompt, max_length=128)
print("\n", output)
